### Steps to take

1. prepare data and clean data
2. select model, fit model and config hyperparameters
3. deploy model
4. OPTIONAL - if deployed model can be turned to a function for Go

In [13]:
import json
import pandas as pd

In [37]:
try:
    with open("bookworms_dataset.json", "r") as file:
        data = json.load(file)
except FileNotFoundError:
    print("Error file not found")
except json.JSONDecodeError:
    print("Error decoding json")

books = pd.DataFrame(data["books"])
users = pd.DataFrame(data["bookworms"])

users

,user_id,name,age,location,member_since,preferences,reading_stats,books_read,wishlist
0,u001,Fadi,29,"{'city': 'Beirut', 'country': 'Lebanon', 'time...",2021-03-15,"{'favorite_genres': ['dystopia', 'feminist', '...","{'total_books_read': 2, 'reading_goal_per_year...","[{'book_id': 'b001', 'rating': 5, 'date_read':...","[{'book_id': 'b014', 'added_date': '2022-06-01..."
1,u002,Peggy,34,"{'city': 'Dublin', 'country': 'Ireland', 'time...",2020-07-01,"{'favorite_genres': ['classic', 'romance', 'dy...","{'total_books_read': 3, 'reading_goal_per_year...","[{'book_id': 'b002', 'rating': 4, 'date_read':...","[{'book_id': 'b012', 'added_date': '2022-03-01..."
2,u003,Amara,26,"{'city': 'Lagos', 'country': 'Nigeria', 'timez...",2022-01-20,"{'favorite_genres': ['fiction', 'historical', ...","{'total_books_read': 4, 'reading_goal_per_year...","[{'book_id': 'b023', 'rating': 5, 'date_read':...","[{'book_id': 'b047', 'added_date': '2022-10-01..."
3,u004,Soren,41,"{'city': 'Copenhagen', 'country': 'Denmark', '...",2019-11-11,"{'favorite_genres': ['sci-fi', 'dystopia', 'cl...","{'total_books_read': 6, 'reading_goal_per_year...","[{'book_id': 'b005', 'rating': 5, 'date_read':...","[{'book_id': 'b042', 'added_date': '2022-05-01..."
4,u005,Lena,31,"{'city': 'Berlin', 'country': 'Germany', 'time...",2021-06-30,"{'favorite_genres': ['fiction', 'drama', 'ment...","{'total_books_read': 5, 'reading_goal_per_year...","[{'book_id': 'b033', 'rating': 5, 'date_read':...","[{'book_id': 'b030', 'added_date': '2022-11-01..."
5,u006,Tomás,38,"{'city': 'Buenos Aires', 'country': 'Argentina...",2020-02-14,"{'favorite_genres': ['thriller', 'crime', 'fic...","{'total_books_read': 4, 'reading_goal_per_year...","[{'book_id': 'b007', 'rating': 5, 'date_read':...","[{'book_id': 'b032', 'added_date': '2022-01-10..."
6,u007,Yuki,27,"{'city': 'Tokyo', 'country': 'Japan', 'timezon...",2022-04-01,"{'favorite_genres': ['fiction', 'sci-fi', 'con...","{'total_books_read': 4, 'reading_goal_per_year...","[{'book_id': 'b016', 'rating': 5, 'date_read':...","[{'book_id': 'b035', 'added_date': '2023-01-25..."
7,u008,Priya,33,"{'city': 'Mumbai', 'country': 'India', 'timezo...",2019-08-08,"{'favorite_genres': ['classic', 'romance', 'co...","{'total_books_read': 6, 'reading_goal_per_year...","[{'book_id': 'b012', 'rating': 5, 'date_read':...","[{'book_id': 'b020', 'added_date': '2022-05-01..."
8,u009,Kwame,45,"{'city': 'Accra', 'country': 'Ghana', 'timezon...",2018-05-22,"{'favorite_genres': ['historical', 'fiction', ...","{'total_books_read': 7, 'reading_goal_per_year...","[{'book_id': 'b022', 'rating': 5, 'date_read':...","[{'book_id': 'b047', 'added_date': '2022-07-01..."
9,u010,Clara,22,"{'city': 'Lisbon', 'country': 'Portugal', 'tim...",2023-01-05,"{'favorite_genres': ['contemporary', 'romance'...","{'total_books_read': 3, 'reading_goal_per_year...","[{'book_id': 'b034', 'rating': 5, 'date_read':...","[{'book_id': 'b019', 'added_date': '2023-03-15..."


In [40]:
from pandas import json_normalize

users_full = json_normalize(data["bookworms"])
users_full.head(3)

,user_id,name,age,member_since,books_read,wishlist,location.city,location.country,location.timezone,preferences.favorite_genres,...,preferences.preferred_moods,preferences.preferred_length,preferences.preferred_era,reading_stats.total_books_read,reading_stats.reading_goal_per_year,reading_stats.books_read_this_year,reading_stats.avg_rating_given,reading_stats.longest_streak_days,reading_stats.most_read_genre,reading_stats.most_read_author_id
0,u001,Fadi,29,2021-03-15,"[{'book_id': 'b001', 'rating': 5, 'date_read':...","[{'book_id': 'b014', 'added_date': '2022-06-01...",Beirut,Lebanon,Asia/Beirut,"[dystopia, feminist, fiction]",...,"[dark, cerebral]",medium,"[modern, contemporary]",2,24,7,4.50,21,dystopia,a001
1,u002,Peggy,34,2020-07-01,"[{'book_id': 'b002', 'rating': 4, 'date_read':...","[{'book_id': 'b012', 'added_date': '2022-03-01...",Dublin,Ireland,Europe/Dublin,"[classic, romance, dystopia]",...,"[romantic, contemplative, dark]",long,"[classic, modern]",3,30,12,4.67,45,classic,a001
2,u003,Amara,26,2022-01-20,"[{'book_id': 'b023', 'rating': 5, 'date_read':...","[{'book_id': 'b047', 'added_date': '2022-10-01...",Lagos,Nigeria,Africa/Lagos,"[fiction, historical, feminist]",...,"[uplifting, cerebral]",medium,"[modern, contemporary]",4,20,5,4.50,14,fiction,a015


In [42]:
rows = []

for user in data["bookworms"]:
    for br in user["books_read"]:
        rows.append({
            "user_id": user["user_id"],
            "book_id": br["book_id"],
            "rating": br["rating"],
            "did_not_finish": br["did_not_finish"],
            "reread": br["reread"]
        })

interactions = pd.DataFrame(rows)

interactions

,user_id,book_id,rating,did_not_finish,reread
0,u001,b001,5,False,False
1,u001,b003,4,False,False
2,u002,b002,4,False,False
3,u002,b001,5,False,True
4,u002,b004,5,False,True
...,...,...,...,...,...
73,u015,b050,4,False,False
74,u015,b005,4,False,False
75,u015,b009,5,False,False
76,u015,b006,3,False,False


In [35]:
df = interactions.merge(books, on="book_id", how="left")

df

,user_id,book_id,rating,did_not_finish,reread,title,author,author_id,genres,year,pages,avg_rating,language,isbn
0,u001,b001,5,False,False,The Handmaid's Tale,Margaret Atwood,a001,"[dystopia, fiction, feminist]",1985,311,4.1,en,9780385490818
1,u001,b003,4,False,False,The Bell Jar,Sylvia Plath,a002,"[fiction, classic, mental-health]",1963,244,4.0,en,9780060837020
2,u002,b002,4,False,False,Oryx and Crake,Margaret Atwood,a001,"[sci-fi, dystopia, fiction]",2003,374,4.0,en,9780385721677
3,u002,b001,5,False,True,The Handmaid's Tale,Margaret Atwood,a001,"[dystopia, fiction, feminist]",1985,311,4.1,en,9780385490818
4,u002,b004,5,False,True,Jane Eyre,Charlotte Brontë,a003,"[classic, romance, fiction]",1847,532,4.2,en,9780141441146
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
73,u015,b050,4,False,False,The Name of the Rose,Umberto Eco,a031,"[historical, thriller, fiction]",1980,502,4.1,en,9780156001311
74,u015,b005,4,False,False,1984,George Orwell,a004,"[dystopia, classic, sci-fi]",1949,328,4.2,en,9780451524935
75,u015,b009,5,False,False,The Road,Cormac McCarthy,a008,"[dystopia, fiction, survival]",2006,287,4.0,en,9780307387899
76,u015,b006,3,False,False,Brave New World,Aldous Huxley,a005,"[dystopia, sci-fi, classic]",1932,311,3.9,en,9780060850524


In [44]:
user_prefs = {
    u["user_id"]: u["preferences"]["favorite_genres"]
    for u in data["bookworms"]
}

user_prefs

{'u001': ['dystopia', 'feminist', 'fiction'],
 'u002': ['classic', 'romance', 'dystopia'],
 'u003': ['fiction', 'historical', 'feminist'],
 'u004': ['sci-fi', 'dystopia', 'classic'],
 'u005': ['fiction', 'drama', 'mental-health'],
 'u006': ['thriller', 'crime', 'fiction'],
 'u007': ['fiction', 'sci-fi', 'contemporary'],
 'u008': ['classic', 'romance', 'coming-of-age'],
 'u009': ['historical', 'fiction', 'drama'],
 'u010': ['contemporary', 'romance', 'fiction'],
 'u011': ['classic', 'historical', 'fiction'],
 'u012': ['sci-fi', 'feminist', 'fiction'],
 'u013': ['historical', 'drama', 'coming-of-age'],
 'u014': ['fiction', 'feminist', 'contemporary'],
 'u015': ['crime', 'thriller', 'fiction']}

In [46]:
def genre_overlap(user_id, book_genres):
    user_genres = set(user_prefs[user_id])
    book_genres = set(book_genres)
    return len(user_genres & book_genres) / len(user_genres | book_genres)

df["genre_overlap"] = df.apply(
    lambda x: genre_overlap(x["user_id"], x["genres"]),
    axis=1
)

df

,user_id,book_id,rating,did_not_finish,reread,title,author,author_id,genres,year,pages,avg_rating,language,isbn,genre_overlap
0,u001,b001,5,False,False,The Handmaid's Tale,Margaret Atwood,a001,"[dystopia, fiction, feminist]",1985,311,4.1,en,9780385490818,1.0
1,u001,b003,4,False,False,The Bell Jar,Sylvia Plath,a002,"[fiction, classic, mental-health]",1963,244,4.0,en,9780060837020,0.2
2,u002,b002,4,False,False,Oryx and Crake,Margaret Atwood,a001,"[sci-fi, dystopia, fiction]",2003,374,4.0,en,9780385721677,0.2
3,u002,b001,5,False,True,The Handmaid's Tale,Margaret Atwood,a001,"[dystopia, fiction, feminist]",1985,311,4.1,en,9780385490818,0.2
4,u002,b004,5,False,True,Jane Eyre,Charlotte Brontë,a003,"[classic, romance, fiction]",1847,532,4.2,en,9780141441146,0.5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
73,u015,b050,4,False,False,The Name of the Rose,Umberto Eco,a031,"[historical, thriller, fiction]",1980,502,4.1,en,9780156001311,0.5
74,u015,b005,4,False,False,1984,George Orwell,a004,"[dystopia, classic, sci-fi]",1949,328,4.2,en,9780451524935,0.0
75,u015,b009,5,False,False,The Road,Cormac McCarthy,a008,"[dystopia, fiction, survival]",2006,287,4.0,en,9780307387899,0.2
76,u015,b006,3,False,False,Brave New World,Aldous Huxley,a005,"[dystopia, sci-fi, classic]",1932,311,3.9,en,9780060850524,0.0


In [47]:
user_authors = {}

for user in data["bookworms"]:
    read_authors = []
    for br in user["books_read"]:
        book = books[books["book_id"] == br["book_id"]].iloc[0]
        read_authors.append(book["author_id"])
    user_authors[user["user_id"]] = set(read_authors)

df["author_affinity"] = df.apply(
    lambda x: 1 if x["author_id"] in user_authors[x["user_id"]] else 0,
    axis=1
)

In [49]:
user_avg = interactions.groupby("user_id")["rating"].mean()

df["user_avg_rating"] = df["user_id"].map(user_avg)
df["rating_delta"] = df["user_avg_rating"] - df["avg_rating"]

In [50]:
df["reread"] = df["reread"].astype(int)
df["did_not_finish"] = df["did_not_finish"].astype(int)

In [51]:
feature_cols = [
    "genre_overlap",
    "author_affinity",
    "rating_delta",
    "reread",
    "did_not_finish",
    "pages",
    "year"
]

X = df[feature_cols]
y = df["rating"]


In [53]:
from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor()
model.fit(X, y)
model.score(X, y)

0.8718581553839163